# Breast Cancer Detection — Enhanced Notebook

This notebook complements the light Gradio demo in `main.py` by providing: 

- A clear walkthrough of data preprocessing and demo usage.
- Quick visualization helpers you can run locally.
- A lightweight image analysis function (heuristic) for immediate feedback.

**Note**: This notebook uses a simple heuristic for demo purposes — replace with your trained PyTorch model when ready.

In [ ]:
from PIL import Image
import numpy as np
import matplotlib.pyplot as plt

def analyze_image(path_or_array):
    if isinstance(path_or_array, str):
        img = Image.open(path_or_array).convert('RGB')
    else:
        img = Image.fromarray(np.uint8(path_or_array)).convert('RGB')
    arr = np.asarray(img)
    mean = arr.mean()
    std = arr.std()
    score = max(0.0, min(1.0, (140 - mean) / 100.0 + std / 255.0))
    if score < 0.35:
        label = 'Likely benign'
    elif score < 0.65:
        label = 'Unclear — recommend follow-up'
    else:
        label = 'Possible malignant — seek medical advice'
    return {'label': label, 'score': score, 'mean': mean, 'std': std} 

In [ ]:
def show_hist(path_or_array):
    if isinstance(path_or_array, str):
        img = Image.open(path_or_array).convert('L')
    else:
        img = Image.fromarray(np.uint8(path_or_array)).convert('L')
    arr = np.asarray(img).ravel()
    plt.figure(figsize=(6,2))
    plt.hist(arr, bins=32, color='#4c72b0')
    plt.title('Pixel intensity distribution')
    plt.show()

## How to use

1. Run the Gradio app locally: `python main.py` and open http://127.0.0.1:7860.
2. Upload images there, or run the helper functions below to analyze images inline in the notebook.

In [ ]:
from PIL import Image, ImageFilter
from IPython.display import display
import numpy as np
import matplotlib.pyplot as plt


def saliency_overlay(path_or_array, alpha=0.5):
    """Compute and display a simple saliency overlay for the given image.

    Usage:
    - `saliency_overlay('/path/to/image.png')`
    - `saliency_overlay(numpy_array)`
    """
    if isinstance(path_or_array, str):
        img = Image.open(path_or_array).convert('RGB')
    else:
        img = Image.fromarray(np.uint8(path_or_array)).convert('RGB')

    gray = np.asarray(img.convert('L'), dtype=np.float32)
    gx = np.zeros_like(gray)
    gy = np.zeros_like(gray)
    gx[:, 1:-1] = gray[:, 2:] - gray[:, :-2]
    gy[1:-1, :] = gray[2:, :] - gray[:-2, :]
    mag = np.sqrt(gx ** 2 + gy ** 2)

    if mag.max() > 0:
        mag = (mag - mag.min()) / (mag.max() - mag.min())
    else:
        mag = mag * 0.0

    mag_img = Image.fromarray((mag * 255).astype(np.uint8)).filter(ImageFilter.GaussianBlur(radius=2))
    cmap = plt.get_cmap('jet')
    mag_arr = np.asarray(mag_img) / 255.0
    colored = (cmap(mag_arr)[:, :, :3] * 255).astype(np.uint8)
    colored_img = Image.fromarray(colored).convert('RGBA')

    blended = Image.blend(img.convert('RGBA'), colored_img, alpha=alpha)
    display(blended)
